# SmolLM2 V12 — Brute-force every Transformer block with ternary {-1,0,+1}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/SmolLM2_360M_V12_All32_BruteForce_Ternary_CeNN_Colab.ipynb)

**32 blocks × 7 projections = 224 modules force-replaced**. This is deliberately **not** V11's quality-gated selection: all replacements are installed regardless of language quality, so the experiment shows whether full conversion works or collapses. The original FP16 teacher remains intact.

The seven projections are attention `q_proj`, `k_proj`, `v_proj`, `o_proj` and MLP `gate_proj`, `up_proj`, `down_proj`. **Embeddings, norms and lm_head stay FP16.** Codes are strictly −1,0,+1 and packed four values per byte, with one FP16 scale per 128 weights.

**Five comparisons:** progressive 8→16→24→32 attention-output sites, all 128 attention projections, all 96 MLP projections, and all **224** in all 32 blocks. Distinct independent WikiText validation/test windows show teacher-token agreement, gold-label perplexity, compressed model state and latency. Quantization is *activation-aware* via true teacher input `E[x²]`; optional brief local QAT runs without a heavy compiler build.

**Important:** The V9/V11 tiled CUDA inference kernel expands each output tile to FP16 and can be considerably slower than dense FP16, even when checkpoint state is much smaller. It is not native low-bit GEMM. Default **no large checkpoint export** when brute-forced model quality is poor; the full evidence ZIP always downloads. No Hugging Face publication. 


In [ ]:
#@title 1. Native Colab GPU + only missing binary dependencies
import os,sys,json,gc,math,zipfile,copy,random,statistics,subprocess,importlib.util
from pathlib import Path
from urllib.request import urlopen
import torch,numpy as np,pandas as pd
from torch import nn
from torch.nn import functional as F
from IPython.display import display
packages=[("transformers","transformers>=4.50,<5"),("datasets","datasets>=2.20,<5"),
          ("huggingface_hub","huggingface_hub>=0.35"),("safetensors","safetensors>=0.4")]
missing=[pkg for module,pkg in packages if importlib.util.find_spec(module) is None]
if missing:subprocess.run([sys.executable,"-m","pip","install","-q","--only-binary=:all:",*missing],check=True)
from transformers import AutoModelForCausalLM,AutoTokenizer
from datasets import load_dataset
from safetensors.torch import save_file
assert torch.cuda.is_available(),"Enable T4/L4/A100 CUDA runtime"
print("GPU",torch.cuda.get_device_name(0),"Torch",torch.__version__)


In [ ]:
#@title 2. Strict all-32 brute force options and pinned model/kernel revisions
SOURCE="HuggingFaceTB/SmolLM2-360M-Instruct" #@param {type:"string"}
SOURCE_SHA="a10cc1512eabd3dde888204e902eca88bddb4951" #@param {type:"string"}
CAL_BATCHES=2 #@param {type:"integer"}
LOCAL_QAT_STEPS=0 #@param {type:"integer"}
LOCAL_QAT_LR=0.008 #@param {type:"number"}
LOCAL_ROWS=64 #@param {type:"integer"}
FEATURE_CENN_GAIN=0.0 #@param {type:"number"}
TILE_ROWS=128 #@param {type:"integer"}
SEQ_LEN=64 #@param {type:"integer"}
EVAL_WINDOWS=4 #@param {type:"integer"}
TOP_POSITIONS=8 #@param {type:"integer"}
RUN_PROGRESSIVE=True #@param {type:"boolean"}
TIME_EVAL=True #@param {type:"boolean"}
BENCH_REPEATS=3 #@param {type:"integer"}
SAVE_UNVALIDATED_ALL224_DELTA=False #@param {type:"boolean"}
OUT=Path("/content/SmolLM2_360M_V12_BruteForce")
OUT.mkdir(parents=True,exist_ok=True)
SEED=2072;torch.manual_seed(SEED);random.seed(SEED);np.random.seed(SEED)
assert CAL_BATCHES>0 and EVAL_WINDOWS>0 and SEQ_LEN>=TOP_POSITIONS>=1
assert 0<=LOCAL_QAT_STEPS<=100 and TILE_ROWS>=16 and BENCH_REPEATS>=2
try:
    from google.colab import userdata
    TOKEN=userdata.get("HF_TOKEN")
except Exception:TOKEN=os.environ.get("HF_TOKEN")
def get_source(sha,filename,alias):
    file=Path("/content")/filename
    with urlopen(f"https://raw.githubusercontent.com/vtavakkoli/TinyCeNN-LM/{sha}/src/tinycenn_lm/{filename}",timeout=90) as response:
        file.write_bytes(response.read())
    spec=importlib.util.spec_from_file_location(alias,str(file))
    mod=importlib.util.module_from_spec(spec)
    sys.modules[alias]=mod;spec.loader.exec_module(mod)
    return mod
V9_SHA="4ea3bf6e5001b7b123d7c137cb09f0ce32cc1c55"
V11_SHA="2e8d7a88a008f7f88e82998c90266fd33d2d5ed2"
V12_SHA="dd2d9c280cb8c9d7add01fb6e91524b038e292c5"
v9=get_source(V9_SHA,"laya_cenn_ternary_concat_v9.py","laya_cenn_ternary_concat_v9")
v11=get_source(V11_SHA,"smol_ternary_cenn_v11.py","smol_ternary_cenn_v11")
v12=get_source(V12_SHA,"smol_ternary_bruteforce_v12.py","smol_ternary_bruteforce_v12")
tok=AutoTokenizer.from_pretrained(SOURCE,revision=SOURCE_SHA,token=TOKEN)
if tok.pad_token_id is None:tok.pad_token=tok.eos_token
teacher=AutoModelForCausalLM.from_pretrained(
    SOURCE,revision=SOURCE_SHA,token=TOKEN,
    torch_dtype=torch.float16,attn_implementation="sdpa").cuda().eval().requires_grad_(False)
assert len(teacher.model.layers)==32
SETS=v12.all_path_sets(teacher)
assert {k:len(v) for k,v in SETS.items()}=={
 "attention_output_only":32,"attention_all":128,
 "mlp_all":96,"all_7_projections_every_layer":224}
ALL_PATHS=sorted(SETS["all_7_projections_every_layer"])
FP16_BYTES=v12.bytes_of_state(teacher)
print("32 blocks, all 224 target projections found.")
print("Baseline FP16 state MiB",round(FP16_BYTES/2**20,2))


In [ ]:
#@title 3. Real WikiText calibration and independent validation/test data
WIKI_REPO="Salesforce/wikitext";WIKI_CONFIG="wikitext-2-raw-v1"
wiki=load_dataset(WIKI_REPO,WIKI_CONFIG)
assert {"train","validation","test"}.issubset(wiki.keys())
def windows(rows,count):
    tokens=[]
    for item in rows:
        text=item["text"].strip()
        if len(text)<70:continue
        tokens.extend(tok.encode(text,add_special_tokens=False))
        tokens.append(tok.eos_token_id)
        if len(tokens)>=(count+2)*(SEQ_LEN+1):break
    data=[tokens[j*(SEQ_LEN+1):(j+1)*(SEQ_LEN+1)] for j in range(count)]
    if any(len(row)!=SEQ_LEN+1 for row in data) or len(data)!=count:
        raise RuntimeError("Insufficient WikiText tokens for fixed windows")
    return data
train=wiki["train"]
cal_text=windows(train.select(range(min(1000,len(train)))),CAL_BATCHES*2)
val_text=windows(wiki["validation"],EVAL_WINDOWS)
test_text=windows(wiki["test"],EVAL_WINDOWS)
def make_batches(rows):
    bs=[]
    for i in range(0,len(rows),2):
        ids=torch.tensor([r[:-1] for r in rows[i:i+2]],device="cuda")
        bs.append({"input_ids":ids,"attention_mask":torch.ones_like(ids)})
    return bs
CAL=make_batches(cal_text)[:CAL_BATCHES]
VAL=make_batches(val_text);TEST=make_batches(test_text)
@torch.inference_mode()
def ref_for(batches):
    return [teacher(**b,use_cache=False).logits[:,-TOP_POSITIONS:,:].float().cpu()
            for b in batches]
REF_VAL=ref_for(VAL);REF_TEST=ref_for(TEST)
BASE_VAL=v12.gold_ppl(teacher,val_text)
BASE_TEST=v12.gold_ppl(teacher,test_text)
print("Independent baseline PPL: validation",round(BASE_VAL["ppl"],3),
      "test",round(BASE_TEST["ppl"],3))


In [ ]:
#@title 4. Capture activation second moments at all 224 teacher projection sites
MOMENTS={};ACTIVATION_ROWS={};handles=[]
def prehook(name):
    def callback(module,args):
        xs=args[0].detach().reshape(-1,args[0].shape[-1]).float()
        second=xs.square().mean(0).cpu()
        if name not in MOMENTS:MOMENTS[name]=[second,1]
        else:
            MOMENTS[name][0]+=second
            MOMENTS[name][1]+=1
        if LOCAL_QAT_STEPS>0 and name not in ACTIVATION_ROWS:
            ACTIVATION_ROWS[name]=xs[:LOCAL_ROWS].half().cpu()
    return callback
for path in ALL_PATHS:
    handles.append(teacher.get_submodule(path).register_forward_pre_hook(prehook(path)))
try:
    with torch.inference_mode():
        for idx,b in enumerate(CAL):
            teacher(**b,use_cache=False)
            print("Calibrated batch",idx+1,"/",len(CAL))
finally:
    for handle in handles:handle.remove()
assert len(MOMENTS)==224
MOMENTS={name:tot/count for name,(tot,count) in MOMENTS.items()}
print("All 224 activation moments recorded.")


In [ ]:
#@title 5. FORCE pack every matrix: all 224, no quality-gated rejection
PACKED={};MODULES=[]
for idx,path in enumerate(ALL_PATHS,1):
    source_linear=teacher.get_submodule(path)
    packed=v12.packed_from_activation_aware(
        source_linear,MOMENTS[path].cuda(),chunk_rows=TILE_ROWS)
    if LOCAL_QAT_STEPS>0:
        # Optional LOCAL-only STE optimization; no end-to-end teacher KL.
        q=v11.TernaryQATLinear(source_linear,rank=0,
                 feature_residual=FEATURE_CENN_GAIN!=0)
        with torch.no_grad():
            codes=v9.unpack_ternary(packed.packed_weight,packed.padded)
            padded=F.pad(source_linear.weight.detach().float(),
                         (0,packed.padded-source_linear.in_features))
            scale=packed.group_scales.float().repeat_interleave(128,dim=1).clamp_min(1e-8)
            latent=codes.float()+((padded/scale)-codes.float()).clamp(-.46,.46)
            q.codes.copy_(latent)
            q.log_scales.copy_(packed.group_scales.float().clamp_min(1e-8).log())
        x=ACTIVATION_ROWS[path].cuda()
        with torch.no_grad():target=source_linear(x).float()
        optimizer=torch.optim.AdamW(q.parameters(),lr=0.008,weight_decay=0)
        for j in range(LOCAL_QAT_STEPS):
            optimizer.zero_grad(set_to_none=True)
            z=q(x).float()
            loss=(z-target).square().mean()/target.square().mean().clamp_min(1e-8)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(q.parameters(),1.)
            optimizer.step()
        packed=q.pack(TILE_ROWS).eval()
        del q,x,target,optimizer
    elif FEATURE_CENN_GAIN!=0:
        packed=v11.PackedCeNNLinear(
            packed,gain=torch.tensor(FEATURE_CENN_GAIN,device="cuda",
                                     dtype=torch.float16)).eval()
    PACKED[path]=packed
    size_before=v12.bytes_of_projection(source_linear)
    size_after=v12.bytes_of_projection(packed)
    MODULES.append({"name":path,"layer":int(path.split(".")[2]),
       "projection":path.rsplit(".",1)[-1],
       "original_bytes":size_before,"packed_bytes":size_after,
       "saving_pct":100*(1-size_after/size_before)})
    if idx%16==0:
        print("Packed",idx,"/224",
              "| projection saving",round(
                  100*(1-sum(m["packed_bytes"] for m in MODULES)/
                      sum(m["original_bytes"] for m in MODULES)),2),"%")
        (OUT/"packing_progress.json").write_text(
            json.dumps({"packed":idx,"count":224,"modules":MODULES},indent=2))
assert len(PACKED)==224
pd.DataFrame(MODULES).to_csv(OUT/"all224_module_compression.csv",index=False)
print("READY: 224 real 2-bit ternary projection buffers.")


In [ ]:
#@title 6. One reusable student with unconditional forced projection modes
student=copy.deepcopy(teacher).eval().requires_grad_(False)
ACTIVE=set();SCENARIOS=[]
def activate(target):
    global ACTIVE
    wanted=set(target)
    for name in sorted(ACTIVE-wanted):
        v12.swap(student,name,teacher.get_submodule(name))
    for name in sorted(wanted-ACTIVE):
        v12.swap(student,name,PACKED[name])
    ACTIVE=wanted
    installed=sum(not isinstance(student.get_submodule(p),nn.Linear)
                  for p in ALL_PATHS)
    assert installed==len(wanted),(installed,len(wanted))
@torch.inference_mode()
def timing(model,batch,repeats=BENCH_REPEATS):
    def forward():return model(**batch,use_cache=False).logits
    forward();torch.cuda.synchronize()
    times=[]
    for _ in range(repeats):
        a=torch.cuda.Event(enable_timing=True)
        b=torch.cuda.Event(enable_timing=True)
        a.record();result=forward();b.record();b.synchronize()
        times.append(float(a.elapsed_time(b)));del result
    return statistics.median(times)
TEACHER_MS=timing(teacher,VAL[0]) if TIME_EVAL else None
def score(stage,ppl=True,latency=True):
    item={"scenario":stage,"replaced_count":len(ACTIVE),
          "attention_replaced":len(ACTIVE&SETS["attention_all"]),
          "mlp_replaced":len(ACTIVE&SETS["mlp_all"]),
          "blocks_touched":len({int(n.split(".")[2]) for n in ACTIVE}),
          "packed_model_state_mib":v12.bytes_of_state(student)/2**20,
          "model_state_saving_pct":100*(1-v12.bytes_of_state(student)/FP16_BYTES)}
    try:
        item.update(v12.teacher_fidelity(student,VAL,REF_VAL,TOP_POSITIONS))
        if ppl:item["validation_ppl"]=v12.gold_ppl(student,val_text)["ppl"]
        if TIME_EVAL and latency:
            item["candidate_ms"]=timing(student,VAL[0])
            item["speedup"]=TEACHER_MS/item["candidate_ms"]
        print("CHECK",stage,"forced",len(ACTIVE),
              "agreement",round(item["teacher_agreement"],4),
              "memory saving",round(item["model_state_saving_pct"],2),"%")
    except Exception as exc:
        item["evaluation_error"]=repr(exc)[:700]
        print("QUALITY FAILED",stage,item["evaluation_error"][:210])
    SCENARIOS.append(item)
    (OUT/"scenario_progress.json").write_text(json.dumps(SCENARIOS,indent=2))
    return item
print("All ternary projections prepared; no candidate quality gates.")


In [ ]:
#@title 7. Brute-force all 32 decoder blocks and compare each projection group
o_order=sorted(SETS["attention_output_only"],
               key=lambda x:int(x.split(".")[2]))
if RUN_PROGRESSIVE:
    for n in (8,16,24,32):
        activate(o_order[:n])
        score(f"progressive_{n}_o_proj",ppl=n==32,latency=False)
else:
    activate(SETS["attention_output_only"])
    score("all_32_attention_o_proj")
activate(SETS["attention_all"])
score("all_128_attention_q_k_v_o")
activate(SETS["mlp_all"])
score("all_96_mlp_gate_up_down")
activate(SETS["all_7_projections_every_layer"])
score("all_224_in_32_layers")
assert len(ACTIVE)==224
df=pd.DataFrame(SCENARIOS)
df.to_csv(OUT/"bruteforce_mode_comparison.csv",index=False)
display(df)
print("All 224 projection weights are currently replaced. Quality was NOT gated.")


In [ ]:
#@title 8. Independent held-out TEST and short chat on fully forced 224 projection model
FULL_TEST={}
try:
    FULL_TEST["teacher_fidelity"]=v12.teacher_fidelity(
        student,TEST,REF_TEST,TOP_POSITIONS)
    FULL_TEST["candidate_gold_ppl"]=v12.gold_ppl(student,test_text)
    FULL_TEST["original_gold_ppl"]=BASE_TEST
    print("ALL224 INDEPENDENT TEST",json.dumps(FULL_TEST,indent=2))
except Exception as err:
    FULL_TEST["error"]=repr(err)[:800]
    print("Held-out comparison failed:",FULL_TEST["error"])
CHAT=[]
for question in ("Explain why the sky is blue.","What is 17 multiplied by 23?"):
    row={"question":question}
    try:
        text=tok.apply_chat_template(
            [{"role":"user","content":question}],tokenize=False,
            add_generation_prompt=True)
        encoded=tok(text,add_special_tokens=False,return_tensors="pt")
        input_ids=encoded["input_ids"].to("cuda")
        mask=encoded.get("attention_mask")
        if mask is None:mask=torch.ones_like(encoded["input_ids"])
        mask=mask.to("cuda")
        for label,model in (("fp16",teacher),("all224",student)):
            with torch.inference_mode():
                generated=model.generate(
                   input_ids=input_ids,attention_mask=mask,
                   max_new_tokens=24,do_sample=False,use_cache=True,
                   pad_token_id=tok.pad_token_id)
            row[label]=tok.decode(generated[0,input_ids.shape[-1]:],
                                  skip_special_tokens=True)
        print("Prompt:",question,"\nOriginal:",row["fp16"],
              "\nAll224:",row["all224"])
    except Exception as err:
        row["generation_error"]=repr(err)[:600]
        print("Chat failed but result export proceeds",row["generation_error"])
    CHAT.append(row)
(OUT/"all224_chat_outputs.json").write_text(json.dumps(CHAT,indent=2))


In [ ]:
#@title 9. Optional raw ALL224 compressed delta; explicit warning if quality collapsed
raw_delta=OUT/"smol360_v12_all224_ternary_delta.safetensors"
if raw_delta.exists():raw_delta.unlink()
if SAVE_UNVALIDATED_ALL224_DELTA:
    tensors={}
    for path in sorted(ACTIVE):
        state=student.get_submodule(path).state_dict()
        for key,value in state.items():
            tensors[path+"."+key]=value.detach().cpu().contiguous()
    save_file(tensors,str(raw_delta))
    print("Experimental, not quality-approved delta saved",
          raw_delta.stat().st_size/2**20,"MiB")
    del tensors
else:
    print("Raw delta skipped by default to avoid downloading a large unusable model. "
          "Set SAVE_UNVALIDATED_ALL224_DELTA=True if needed.")


In [ ]:
#@title 10. Full evidence ZIP — every layer, every mode, storage and quality
report={
 "experiment":"SmolLM2 V12 brute-force strict ternary {-1,0,+1}",
 "source_model":SOURCE,"source_sha":SOURCE_SHA,
 "gpu":torch.cuda.get_device_name(0),"torch":str(torch.__version__),
 "transformer_layers":32,
 "projections_forced":224,
 "conversion":"activation-aware signed 2-bit G128, no acceptance gate",
 "optional_local_qat_steps":LOCAL_QAT_STEPS,
 "optional_feature_cenn_gain":FEATURE_CENN_GAIN,
 "fp16_model_state_mib":FP16_BYTES/2**20,
 "teacher_validation_ppl":BASE_VAL,"teacher_test_ppl":BASE_TEST,
 "scenario_results":SCENARIOS,"all224_test":FULL_TEST,
 "chat_samples":CHAT,
 "full224_raw_delta_saved":raw_delta.exists(),
 "full224_raw_delta_mib":raw_delta.stat().st_size/2**20 if raw_delta.exists() else None,
 "disclaimers":[
    "All 7 projections per each of 32 blocks are forced, not embeddings/norms/head.",
    "No accepted-layer quality gating: model may be unusable after full conversion.",
    "Packed 2-bit weights are expanded in FP16 output tiles at inference.",
    "Teacher, student, packed-candidate cache co-reside during CUDA benchmarking.",
    "This run is not a native ternary tensor core acceleration experiment.",
    "Validation/test windows are short; no claim of comprehensive quality assessment."]}
(OUT/"v12_all32_bruteforce_report.json").write_text(json.dumps(report,indent=2))
manifest={"source":SOURCE,"source_sha":SOURCE_SHA,
 "ternary_codec_sha":V9_SHA,"local_qat_runtime_sha":V11_SHA,
 "conversion_runtime_sha":V12_SHA,"path_count":len(ALL_PATHS),
 "paths":ALL_PATHS,"exact_ternary_coefficients":[-1,0,1],
 "requires_original_fp16_base":True}
(OUT/"all224_manifest.json").write_text(json.dumps(manifest,indent=2))
archive=OUT/"SmolLM2_360M_V12_All32_BruteForce_Ternary_Results.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as handle:
    for entry in OUT.iterdir():
        if entry.is_file() and entry!=archive:handle.write(entry,entry.name)
print("Results ZIP",archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:pass


## What to expect from full brute force

The notebook **forces** 224 strict-ternary projection substitutions, unlike V11 (0 accepted). It compares partial attention, full attention, full MLP, and the entire 32-block network, with progression checkpoints. A lower checkpoint size is real only for packed ternary *weights*; if gold perplexity collapses or generated text becomes incoherent, the fully converted model is not successful.

If you want to test training across all 224 modules, rerun with `LOCAL_QAT_STEPS=4` after the fast baseline. This may take substantially longer on T4. Both the basic and QAT paths discard FP32 master training weights after packing; full-model teacher-logit distillation is **not** performed in V12.

The output ZIP contains the full per-module storage table, all scenario quality/latency results and independent held-out test. Only enable raw checkpoint export if you deliberately want a likely degraded all-layer model. No Hugging Face model is modified.
